# ۱ — بارگذاری دو دیتاست و استخراج ویژگی‌های عددی

**پروژه:** Detecting Silent Failures of Uncertainty Estimators under Distribution Shift  
**دیتاست‌ها:** CommonsenseQA (`tau/commonsense_qa`) و ARC-Challenge (`allenai/ai2_arc`, `ARC-Challenge`)

**هدف:** خواندن فایل‌های Parquet که قبلاً دانلود کرده‌ای و ساختن جدول ویژگی‌های عددی، بدون دست‌کاری فایل اصلی و بدون اجرای LLM.

**ویژگی‌ها:** تعداد کلمات سؤال، تعداد کاراکترهای سؤال، میانگین تعداد کلمات گزینه‌ها و تعداد گزینه‌ها. این‌ها برای آمار توصیفی مناسب‌اند، ولی «تعداد کلمات» **معادل تعداد توکن‌های tokenizer یک LLM نیست**. برای `p_max`، آنتروپی و Margin بعداً باید مدل اجرا شود.

ساختار پوشه مورد انتظار:

```text
project/
├── 01_prepare_numeric_features.ipynb
├── 02_statistics_and_distributions.ipynb
├── 03_three_normalizations.ipynb
└── datasets/
    ├── commonsense_qa/   (تمام فایل‌های Parquet مربوط به train/validation/test)
    └── arc_challenge/    (فقط config ARC-Challenge با هر سه split)
```

فایل‌ها می‌توانند در زیرپوشه‌های این دو مسیر باشند. اگر مسیر متفاوت است، مقدار `DATASET_DIRS` را تغییر بده.

In [1]:
# Setup for Google Colab (runs automatically if inside Colab)
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    if not Path('/content/drive').exists():
        drive.mount('/content/drive')
        print("Google Drive mounted at /content/drive")


## نصب کتابخانه‌ها

اگر قبلاً نصب نکرده‌ای، سلول زیر را یک بار بدون `#` اجرا کن.

In [2]:
# %pip install -q pandas numpy pyarrow matplotlib scipy

## ۱. تعیین مسیر فایل‌ها

In [3]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
from IPython.display import display

# Automatic project root detection (works in local project root, notebooks directory, or Google Colab)
def find_project_root():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path('/content/drive/MyDrive/project'),
        Path('/content/drive/MyDrive/Detecting Silent Failures of Uncertainty Estimators'),
        Path('/content/drive/MyDrive'),
        Path('/content'),
    ]
    for cand in candidates:
        if (cand / 'datasets' / 'commonsense_qa').exists() and (cand / 'datasets' / 'arc_challenge').exists():
            return cand
    for cand in candidates:
        if (cand / 'datasets').exists():
            return cand
    return Path.cwd()

ROOT = find_project_root()
DATASET_DIRS = {
    'CommonsenseQA': ROOT / 'datasets' / 'commonsense_qa',
    'ARC-Challenge': ROOT / 'datasets' / 'arc_challenge',
}
OUTPUT = ROOT / 'outputs'
OUTPUT.mkdir(parents=True, exist_ok=True)
print(f"Project ROOT: {ROOT}")
print(f"Output directory: {OUTPUT}")

for name, folder in DATASET_DIRS.items():
    files = sorted(folder.rglob('*.parquet')) if folder.exists() else []
    print(f'{name}: {len(files)} Parquet file(s) at {folder}')
    for f in files:
        print('  ', f.relative_to(folder), f'({f.stat().st_size/1024**2:.3f} MiB)')
    if not files:
        raise FileNotFoundError(f'No Parquet files for {name}. Update DATASET_DIRS: {folder}')


Project ROOT: D:\app\ai\Detecting Silent Failures of Uncertainty Estimators
Output directory: D:\app\ai\Detecting Silent Failures of Uncertainty Estimators\outputs
CommonsenseQA: 3 Parquet file(s) at D:\app\ai\Detecting Silent Failures of Uncertainty Estimators\datasets\commonsense_qa
   data\test-00000-of-00001.parquet (0.144 MiB)
   data\train-00000-of-00001.parquet (1.189 MiB)
   data\validation-00000-of-00001.parquet (0.153 MiB)
ARC-Challenge: 3 Parquet file(s) at D:\app\ai\Detecting Silent Failures of Uncertainty Estimators\datasets\arc_challenge
   ARC-Challenge\test-00000-of-00001.parquet (0.194 MiB)
   ARC-Challenge\train-00000-of-00001.parquet (0.181 MiB)
   ARC-Challenge\validation-00000-of-00001.parquet (0.053 MiB)


## ۲. بارگذاری train / validation / test

هر فایل شناسایی و یکجا بارگذاری می‌شود. از نام فایل یا پوشه، نام split استخراج می‌کنیم. اگر اسامی دانلودشده استاندارد نباشند، خطای واضح دریافت می‌کنی.

In [4]:
def detect_split(path: Path) -> str:
    for part in reversed(path.parts):
        s = part.lower()
        if re.search(r'(?:^|[._/-])validation(?:[._/-]|$)', s):
            return 'validation'
        if re.search(r'(?:^|[._/-])train(?:[._/-]|$)', s):
            return 'train'
        if re.search(r'(?:^|[._/-])test(?:[._/-]|$)', s):
            return 'test'
    raise ValueError(f'Could not identify split from file/path: {path}')

raw = {}
manifest = []
for dataset_name, folder in DATASET_DIRS.items():
    fragments=[]
    for file in sorted(folder.rglob('*.parquet')):
        split=detect_split(file.relative_to(folder))
        frame=pd.read_parquet(file).copy()
        frame['split']=split
        frame['source_file']=str(file.relative_to(folder))
        frame['source_row']=np.arange(len(frame))
        fragments.append(frame)
        manifest.append({'dataset':dataset_name,'split':split,
                         'file':str(file.relative_to(folder)),'rows':len(frame),
                         'size_bytes':file.stat().st_size})
    if not fragments:
        raise FileNotFoundError(f'No files in {folder}')
    raw[dataset_name]=pd.concat(fragments,ignore_index=True,sort=False)

manifest_df=pd.DataFrame(manifest)
display(manifest_df)
display(manifest_df.pivot_table(index='dataset',columns='split',values='rows',aggfunc='sum',fill_value=0))
manifest_df.to_csv(OUTPUT/'dataset_manifest.csv', index=False)
for name,frame in raw.items():
    assert {'question','choices'}.issubset(frame.columns), f'{name}: missing question/choices'
    print(name,len(frame),'rows')

,dataset,split,file,rows,size_bytes
0,CommonsenseQA,test,data\test-00000-of-00001.parquet,1140,151227
1,CommonsenseQA,train,data\train-00000-of-00001.parquet,9741,1247103
2,CommonsenseQA,validation,data\validation-00000-of-00001.parquet,1221,160240
3,ARC-Challenge,test,ARC-Challenge\test-00000-of-00001.parquet,1172,203808
4,ARC-Challenge,train,ARC-Challenge\train-00000-of-00001.parquet,1119,189909
5,ARC-Challenge,validation,ARC-Challenge\validation-00000-of-00001.parquet,299,55743


split,test,train,validation
dataset,,,
ARC-Challenge,1172,1119,299
CommonsenseQA,1140,9741,1221


CommonsenseQA 12102 rows
ARC-Challenge 2590 rows


## ۳. استخراج ویژگی‌ها

توجه: روش شمارش کلمات مستقل از مدل است و صرفاً معیار ساده‌ای برای تحلیل اولیه متن انگلیسی محسوب می‌شود. اگر بعدها تعداد **توکن مدل** لازم بود باید tokenizer خود مدل را اجرا کنیم.

In [5]:
WORD_RE = re.compile(r"[A-Za-z0-9]+(?:['’-][A-Za-z0-9]+)*")

def word_count(text) -> int:
    if text is None:
        return 0
    return len(WORD_RE.findall(str(text)))

def get_option_texts(choices):
    if isinstance(choices, dict):
        items=choices.get('text', [])
    elif isinstance(choices,(list,tuple,np.ndarray)):
        items=choices
    else:
        return []
    if isinstance(items,np.ndarray):
        items=items.tolist()
    if not isinstance(items,(list,tuple)):
        return []
    result=[]
    for val in items:
        if isinstance(val,dict):
            val=val.get('text','')
        if val is not None:
            result.append(str(val))
    return result

def make_features(dataset_name,frame):
    text=frame['question'].fillna('').astype(str)
    options=frame['choices'].map(get_option_texts)
    return pd.DataFrame({
        'dataset':dataset_name,
        'split':frame['split'].to_numpy(),
        'source_file':frame['source_file'].to_numpy(),
        'source_row':frame['source_row'].to_numpy(),
        'question_words':text.map(word_count).to_numpy(),
        'question_chars':text.map(len).to_numpy(),
        'avg_option_words':options.map(
            lambda opts: float(np.mean([word_count(t) for t in opts])) if opts else np.nan
        ).to_numpy(),
        'option_count':options.map(len).to_numpy(),
    })

features=pd.concat([make_features(name,frame) for name,frame in raw.items()],ignore_index=True)
FEATURES=['question_words','question_chars','avg_option_words','option_count']
display(features.head())
print('Missing values by dataset:')
display(features.groupby('dataset')[FEATURES].apply(lambda frame:frame.isna().sum()))
print('Rows with no choices:',(features['option_count']==0).sum())

,dataset,split,source_file,source_row,question_words,question_chars,avg_option_words,option_count
0,CommonsenseQA,test,data\test-00000-of-00001.parquet,0,18,85,1.6,5
1,CommonsenseQA,test,data\test-00000-of-00001.parquet,1,13,64,1.4,5
2,CommonsenseQA,test,data\test-00000-of-00001.parquet,2,14,65,1.6,5
3,CommonsenseQA,test,data\test-00000-of-00001.parquet,3,9,49,2.0,5
4,CommonsenseQA,test,data\test-00000-of-00001.parquet,4,9,51,1.2,5


Missing values by dataset:


,question_words,question_chars,avg_option_words,option_count
dataset,,,,
ARC-Challenge,0,0,0,0
CommonsenseQA,0,0,0,0


Rows with no choices: 0


## ۴. ذخیره خروجی این مرحله

In [6]:
target=OUTPUT/'numeric_features.csv'
features.to_csv(target,index=False,encoding='utf-8-sig')
print('Saved:',target)
print('Rows:',len(features))
print('Original Parquet files were NOT modified.')

Saved: D:\app\ai\Detecting Silent Failures of Uncertainty Estimators\outputs\numeric_features.csv
Rows: 14692
Original Parquet files were NOT modified.
